In [0]:
import requests, json

# Get workspace URL and token
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()
token = ctx.apiToken().get()

headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
base_path = "/Users/labuser13955035_1772876383@vocareum.com/H2"

# Current cluster ID for the job
cluster_id = spark.conf.get("spark.databricks.clusterUsageTags.clusterId")

# Delete previous job if it exists
try:
    resp = requests.post(f"{host}/api/2.1/jobs/delete", headers=headers, json={"job_id": 1003516450501965})
    if resp.status_code == 200:
        print("\u2705 Deleted old job (1003516450501965)")
    else:
        print(f"\u26a0\ufe0f  Old job not found or already deleted (status: {resp.status_code})")
except:
    pass

job_config = {
    "name": "H2 Complete ML Pipeline",
    "tasks": [
        # ======== Phase 1: Data Engineering (Sequential) ========
        {
            "task_key": "T01_config",
            "notebook_task": {"notebook_path": f"{base_path}/00_config"},
            "existing_cluster_id": cluster_id
        },
        {
            "task_key": "T02_bootstrap",
            "depends_on": [{"task_key": "T01_config"}],
            "notebook_task": {"notebook_path": f"{base_path}/01_uc_bootstrap"},
            "existing_cluster_id": cluster_id
        },
        {
            "task_key": "T03_bronze_ingest",
            "depends_on": [{"task_key": "T02_bootstrap"}],
            "notebook_task": {"notebook_path": f"{base_path}/02_bronze_ingest"},
            "existing_cluster_id": cluster_id
        },
        {
            "task_key": "T04_silver_clean",
            "depends_on": [{"task_key": "T03_bronze_ingest"}],
            "notebook_task": {
                "notebook_path": f"{base_path}/03_silver_clean",
                "base_parameters": {
                    "start_date": "2020-01-01",
                    "end_date": "2022-01-01",
                    "zone": "NL",
                    "run_id": ""
                }
            },
            "existing_cluster_id": cluster_id
        },
        {
            "task_key": "T05_gold_marts",
            "depends_on": [{"task_key": "T04_silver_clean"}],
            "notebook_task": {"notebook_path": f"{base_path}/04_gold_marts"},
            "existing_cluster_id": cluster_id
        },

        # ======== Phase 2: Feature Pipeline (Sequential) ========
        {
            "task_key": "T06_feature_selection",
            "depends_on": [{"task_key": "T05_gold_marts"}],
            "notebook_task": {"notebook_path": f"{base_path}/05_feature_selection"},
            "existing_cluster_id": cluster_id
        },
        {
            # Pure feature engineering: lags, rolling stats, interactions
            # Saves h2_gold_model_scoring_engineered for downstream models
            "task_key": "T07_advanced_features",
            "depends_on": [{"task_key": "T06_feature_selection"}],
            "notebook_task": {"notebook_path": f"{base_path}/14_AdvancedFeatures_Tuning"},
            "existing_cluster_id": cluster_id
        },

        # ======== Phase 3: ML Models (Parallel) ========
        {
            # Classification: reads h2_gold_model_scoring_engineered
            "task_key": "T08_classification_mlflow",
            "depends_on": [{"task_key": "T07_advanced_features"}],
            "notebook_task": {"notebook_path": f"{base_path}/09_MLflow_Tracking"},
            "existing_cluster_id": cluster_id
        },
        {
            # Regression: reads h2_gold_model_scoring_engineered
            "task_key": "T09_regression",
            "depends_on": [{"task_key": "T07_advanced_features"}],
            "notebook_task": {"notebook_path": f"{base_path}/11_Regression_PricePrediction"},
            "existing_cluster_id": cluster_id
        },
        {
            # Prophet: univariate (timestamp + price only)
            # Reads model_scoring_base directly — no engineered features needed
            "task_key": "T10_timeseries",
            "depends_on": [{"task_key": "T05_gold_marts"}],
            "notebook_task": {"notebook_path": f"{base_path}/12_TimeSeries_Forecasting"},
            "existing_cluster_id": cluster_id
        },
        {
            # ALS recommendation: reads h2_gold_model_scoring_engineered
            "task_key": "T11_recommendation_als",
            "depends_on": [{"task_key": "T07_advanced_features"}],
            "notebook_task": {"notebook_path": f"{base_path}/18_RecommendationSystem_ALS"},
            "existing_cluster_id": cluster_id
        },

        # ======== Phase 4: Production (depends on classification model) ========
        {
            "task_key": "T12_production_pipeline",
            "depends_on": [{"task_key": "T08_classification_mlflow"}],
            "notebook_task": {"notebook_path": f"{base_path}/10_Production_Pipeline"},
            "existing_cluster_id": cluster_id
        },

        # ======== Phase 5: Final System (waits for everything) ========
        {
            "task_key": "T13_final_production",
            "depends_on": [
                {"task_key": "T08_classification_mlflow"},
                {"task_key": "T09_regression"},
                {"task_key": "T10_timeseries"},
                {"task_key": "T11_recommendation_als"},
                {"task_key": "T12_production_pipeline"}
            ],
            "notebook_task": {"notebook_path": f"{base_path}/21_Final_Production_System"},
            "existing_cluster_id": cluster_id
        }
    ],
    "format": "MULTI_TASK"
}

resp = requests.post(f"{host}/api/2.1/jobs/create", headers=headers, json=job_config)
result = resp.json()

if "job_id" in result:
    print(f"\n\u2705 Job created successfully!")
    print(f"   Job ID: {result['job_id']}")
    print(f"   Name:   H2 Complete ML Pipeline")
    print(f"\n   Corrected DAG (13 tasks):")
    print(f"")
    print(f"   T01 config")
    print(f"    \u2192 T02 bootstrap")
    print(f"     \u2192 T03 bronze_ingest")
    print(f"      \u2192 T04 silver_clean")
    print(f"       \u2192 T05 gold_marts \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u252c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510")
    print(f"        \u2192 T06 feature_selection     \u2502  T10 timeseries (Prophet)  \u2502")
    print(f"         \u2192 T07 advanced_features   \u2502  (univariate, no eng feat) \u2502")
    print(f"            \u251c\u2500\u2192 T08 classification  \u2502                              \u2502")
    print(f"            \u251c\u2500\u2192 T09 regression     \u2502                              \u2502")
    print(f"            \u2514\u2500\u2192 T11 recommendation \u2502                              \u2502")
    print(f"                 \u2502                  \u2502                              \u2502")
    print(f"            T08 \u2192 T12 production    \u2502                              \u2502")
    print(f"                 \u2502                  \u2502                              \u2502")
    print(f"            All \u2192 T13 final_production \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518")
else:
    print(f"\u274c Error: {json.dumps(result, indent=2)}")